In [331]:
import numpy as np
from typing import Tuple, List
from copy import deepcopy

# Complex Numbers

In [332]:
3+1j

(3+1j)

In [333]:
(3+1j)+(2+3j), (3+2)+(1j+3j)

((5+4j), (5+4j))

In [334]:
(3+1j)*(2+3j), (3*2)+(3*3j)+(1j*2)+(1j*3j)

((3+11j), (3+11j))

In [335]:
(3+2j).conjugate()

(3-2j)

In [336]:
def magnitude(z: complex) -> float:
    return np.sqrt(z.real**2 +z.imag**2)

In [337]:
magnitude(3+2j)

np.float64(3.605551275463989)

In [338]:
def polar(z: complex) -> Tuple[float, float]:
    r = magnitude(z)
    theta = np.arctan(z.imag/z.real)
    return r, theta

In [339]:
polar(3+2j)

(np.float64(3.605551275463989), np.float64(0.5880026035475675))

# Matrices

In [340]:
a = np.matrix([[1, 2], [3, 4]])
b = np.matrix([[0, 2], [3, 1]])

In [341]:
2*a

matrix([[2, 4],
        [6, 8]])

In [342]:
a+b

matrix([[1, 4],
        [6, 5]])

In [343]:
a*b

matrix([[ 6,  4],
        [12, 10]])

In [344]:
a.__invert__()

matrix([[-2, -3],
        [-4, -5]])

In [345]:
a.T

matrix([[1, 3],
        [2, 4]])

# Matrices complejas

In [346]:
def unitary(m: np.matrix)->np.matrix:
    a: np.matrix = deepcopy(m)
    for i in range(len(m)):
        for k in range(len(m[i])):
            a[i][k] = m[i][k].conjugate()
    return a


In [347]:
a = np.matrix([[2+3j, complex(0)], [complex(5), 3+1j]])

In [348]:
unitary(a)

matrix([[2.-3.j, 0.-0.j],
        [5.-0.j, 3.-1.j]])

In [349]:
def hermitian(m:np.matrix)->np.matrix:
    return unitary(m).T

In [350]:
hermitian(a)

matrix([[2.-3.j, 5.-0.j],
        [0.-0.j, 3.-1.j]])

In [351]:
def is_unitary(m: np.matrix)->bool:
    return m*hermitian(m) == np.matrix(np.eye(m.shape[0], m.shape[1]))

In [352]:
def is_hermitian(m: np.matrix)->bool:
    return m == hermitian(m)

# Qubits

In [353]:
class Qubit:
    def __init__(self, factors: List[complex])->None:
        assert round(np.sqrt(sum(list(map(lambda x: magnitude(x)**2, factors))))) == 1
        self.vector = np.array(factors)
    @property
    def predict(self) -> List[float]:
        return list(map(lambda x: magnitude(x)**2, self.vector))
    
    @property
    def measure(self)->str:
        n = np.random.choice(len(self.vector), p=self.predict)
        return str('|' + bin(n).replace('0b', '').zfill(int(np.log2(len(self.vector)))) + '⟩')

In [354]:
q = Qubit([complex(np.sqrt(3)/2), complex(1/2)])

In [355]:
q.measure

'|0⟩'

# Gates

In [356]:
q.vector

array([0.8660254+0.j, 0.5      +0.j])

## X gate

In [357]:
x_gate = np.matrix([[complex(0), complex(1)], [complex(1), complex(0)]])

In [358]:
q.vector*x_gate

matrix([[0.5      +0.j, 0.8660254+0.j]])

## Y gate

In [359]:
y_gate = np.matrix([[complex(0), complex(0, -1)], [complex(0, 1), complex(0)]])

In [360]:
q.vector*y_gate

matrix([[0.+0.5j      , 0.-0.8660254j]])

## Z gate

In [361]:
z_gate = np.matrix([[complex(1), complex(0)], [complex(0), complex(-1)]])

In [362]:
q.vector*z_gate

matrix([[ 0.8660254+0.j, -0.5      +0.j]])

## Hadamard

In [363]:
H = (1/np.sqrt(2))*np.matrix([[complex(1), complex(1)], [complex(1), complex(-1)]])

In [364]:
Qubit([1, 0]).vector*H

matrix([[0.70710678+0.j, 0.70710678+0.j]])

## S gate

In [365]:
s_gate = np.matrix([[complex(1), complex(0)], [complex(0), complex(0, -1)]])

In [366]:
q.vector*s_gate

matrix([[0.8660254+0.j , 0.       -0.5j]])

## T gate

In [367]:
aux = complex(np.sqrt(2)/2, np.sqrt(2)/2)
t_gate = np.matrix([[complex(1), complex(0)], [complex(0), aux]])

In [368]:
q.vector*t_gate

matrix([[0.8660254 +0.j        , 0.35355339+0.35355339j]])

# Tensor product

In [369]:
def tensor_product(q1: Qubit, q2: Qubit)->Qubit:
    factors: List[complex] = []
    for i in q1.vector:
        for j in q2.vector:
            factors.append(i*j)
    return Qubit(factors)

In [370]:
q2 = tensor_product(q, Qubit([complex(1/np.sqrt(2)), complex(1/np.sqrt(2))]))
q2.vector

array([0.61237244+0.j, 0.61237244+0.j, 0.35355339+0.j, 0.35355339+0.j])

In [371]:
q2.measure

'|00⟩'

# Ejemplo circuito cuantico

![image.png](image.1.png)

este circuito cambia la probabilidad del 100% de medir |001⟩ a una de 50% de |110⟩ y 50% |100⟩

In [372]:
I = np.matrix(np.eye(2, 2))

In [373]:
q1 = Qubit([1, 0])
q2 = Qubit([1, 0])
q3 = Qubit([0, 1])

In [374]:
psi0 = tensor_product(tensor_product(q1, q2), q3)

psi0.vector = np.array(psi0.vector * np.kron(np.kron(I, x_gate), I)).flatten()

psi0.vector = np.array(psi0.vector * np.kron(np.kron(x_gate, H), x_gate)).flatten()

In [375]:
psi0.vector

array([ 0.        +0.j,  0.        +0.j,  0.        +0.j,  0.        +0.j,
        0.70710678+0.j,  0.        +0.j, -0.70710678+0.j,  0.        +0.j])

In [376]:
psi0.measure

'|110⟩'

# Compuertas de mas de un qubit

## CNOT

In [377]:
cnot = np.matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])

In [378]:
aux = tensor_product(Qubit([0, 1]), Qubit([0, 1]))
Qubit(list(np.array(aux.vector * cnot).flatten())).measure

'|10⟩'

## Controled gates

In [379]:
def gen_cgate(gate: np.matrix, lvl: int)->np.matrix:
    I = np.eye(2**lvl, dtype=complex)
    I[-2:, -2:] = gate
    return np.matrix(I)

ejemplo controlled z_gate

In [380]:
cz = gen_cgate(z_gate, 2)

In [381]:
aux = tensor_product(Qubit([0, 1]), Qubit([0, 1]))
Qubit(list(np.array(aux.vector * cz).flatten())).vector

array([ 0.+0.j,  0.+0.j,  0.+0.j, -1.+0.j])

In [382]:
Qubit([0, 1]).vector * z_gate

matrix([[ 0.+0.j, -1.+0.j]])

## Toffoli gate

In [383]:
tg = gen_cgate(x_gate, 3)

In [384]:
q1 = tensor_product(tensor_product(Qubit([0, 1]), Qubit([0, 1])), Qubit([0, 1]))
Qubit(list(np.array(q1.vector * tg).flatten())).measure

'|110⟩'

## swap gate

In [385]:
sw_gate = np.matrix([
    [1, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1]
])

In [386]:
q1 = tensor_product(Qubit([0, 1]), Qubit([1, 0]))
Qubit(list(np.array(q1.vector * sw_gate).flatten())).measure

'|01⟩'

# Nuevo circuito cuantico

![image.png](image.png)

cambia las probabilidades del sistema de |00⟩ a 0.5|00⟩ + 0.5|11⟩

In [387]:
psi0 = tensor_product(Qubit([1, 0]), Qubit([1, 0]))

psi0.vector = np.array(psi0.vector * np.kron(H, I)).flatten()

psi0.vector = np.array(psi0.vector * cnot).flatten()

In [398]:
psi0.measure

'|11⟩'

# Superdense coding

![image.png](image.2.png)

In [418]:
def transmit(b1: int, b2: int)-> Qubit:
    psi0 = tensor_product(Qubit([1, 0]), Qubit([1, 0]))

    psi0.vector = np.array(psi0.vector * np.kron(H, I)).flatten()

    psi0.vector = np.array(psi0.vector * cnot).flatten()

    if b2 == 1:
        psi0.vector = np.array(psi0.vector * np.kron(x_gate, I)).flatten()
    if b1 == 1:
        psi0.vector = np.array(psi0.vector * np.kron(z_gate, I)).flatten()
    
    psi0.vector = np.array(psi0.vector * cnot).flatten()

    psi0.vector = np.array(psi0.vector * np.kron(H, I)).flatten()

    return psi0

In [419]:
transmit(0, 1).measure

'|01⟩'

# Operaciones logicas

## Not gate

![image.png](image.6.png)

In [420]:
def not_o(q: Qubit) -> Qubit:
    return Qubit(list(np.array(q.vector * x_gate).flatten())) 

In [434]:
not_o(Qubit([1, 0])).measure

'|1⟩'

## AND gate

![image.png](image.3.png)

In [424]:
def and_o(q1: Qubit, q2: Qubit) -> Qubit:
    psi0 = tensor_product(q1, q2)
    psi0 = tensor_product(psi0, Qubit([1, 0]))
    psi0.vector = np.array(psi0.vector * tg).flatten()
    return psi0

In [435]:
'|' + and_o(Qubit([0, 1]), Qubit([0, 1])).measure[-2] + '⟩'

'|1⟩'

## XOR gate

![image.png](image.4.png)

In [465]:
def xor(q1: Qubit, q2: Qubit) -> Qubit:
    psi0 = tensor_product(q1, q2)
    psi0.vector = np.array(psi0.vector * cnot).flatten()
    psi0.vector = np.array(psi0.vector * sw_gate).flatten()
    psi0.vector = np.array(psi0.vector * cnot).flatten()
    psi0.vector = np.array(psi0.vector * sw_gate).flatten()
    return psi0

In [ ]:
'|' + xor(Qubit([1, 0]), Qubit([0, 1])).measure[-2] + '⟩'

'|0⟩'

## OR gate


![image.png](image.5.png)

In [460]:
def or_o(q1: Qubit, q2: Qubit) -> Qubit:
    psi0 = tensor_product(q1, q2)
    psi0 = tensor_product(psi0, Qubit([1, 0]))
    psi0.vector = np.array(psi0.vector * np.kron(np.kron(x_gate, x_gate), I)).flatten()
    psi0.vector = np.array(psi0.vector * tg).flatten()
    psi0.vector = np.array(psi0.vector * np.kron(np.kron(I, I), x_gate)).flatten()
    return psi0

In [464]:
'|' + or_o(Qubit([1, 0]), Qubit([1, 0])).measure[-2] + '⟩'

'|0⟩'